# 01. LangGraph를 위한 Python 기초

> LangGraph 코드는 `TypedDict` · `Annotated` · `add_messages` 같은 Python 타이핑 기능을 적극 사용해요. 본격적인 그래프 학습 전에 필요한 Python 기초만 모아 빠르게 점검해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `TypedDict`와 일반 `dict`의 차이를 설명하고, LangGraph에서 State를 정의할 때 TypedDict를 사용하는 이유를 이해할 수 있어요
2. `Annotated`를 사용하여 타입 힌트에 메타데이터를 붙이고, LangGraph의 리듀서(reducer)와 연결하는 방법을 알 수 있어요
3. Pydantic `BaseModel`과 `Field`로 데이터 유효성 검사(validation)를 구현할 수 있어요
4. `add_messages` 리듀서의 동작 원리(추가 vs. 교체)를 이해하고 직접 사용할 수 있어요

## 사전 지식

- Python 딕셔너리(`dict`)와 클래스(`class`) 기본 문법
- 이전 노트북: `01_Introduction/02-Product-Hierarchy.ipynb` — LangChain vs LangGraph vs Deep Agents 제품 계층

## 왜 이 문법을 배워야 할까요?

LangGraph로 에이전트를 만들 때 우리는 반드시 **State(상태)**를 정의해야 해요. 이 State는 그래프가 실행되는 동안 노드 간에 데이터를 전달하는 역할을 해요.

State를 올바르게 정의하려면 세 가지 Python 도구를 알아야 해요:

| 도구 | 역할 | LangGraph에서 사용 위치 |
|------|------|------------------------|
| `TypedDict` | 딕셔너리의 키/타입 구조를 고정 | State 클래스 정의 |
| `Annotated` | 타입에 메타데이터(리듀서) 첨부 | `messages: Annotated[list, add_messages]` |
| `Pydantic BaseModel` | 입력 데이터 유효성 검사 | 노드의 입력/출력 스키마 |
| `add_messages` | 메시지 리스트를 스마트하게 병합 | State의 messages 필드 리듀서 |

아래 다이어그램에서 이 도구들이 LangGraph 안에서 어떻게 연결되는지 살펴볼게요:

```mermaid
flowchart TD
    A["Python TypedDict<br>상태 구조 정의"] --> D
    B["Python Annotated<br>리듀서 메타데이터 첨부"] --> D
    C["Pydantic BaseModel<br>입력값 유효성 검사"] --> E
    D["LangGraph State<br>class State(TypedDict):<br>  messages: Annotated[list, add_messages]"] --> F
    E["노드 입력/출력 스키마"] --> F
    F["StateGraph 컴파일 및 실행"]
    G["add_messages 리듀서<br>메시지 추가 또는 교체"] --> D

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A,B,C,G input
    class D,E process
    class F output
```

---

## 1. TypedDict — 구조가 있는 딕셔너리

Python의 일반 `dict`는 어떤 키든 자유롭게 추가/변경할 수 있어요. 이건 편리하지만, 코드가 복잡해지면 실수가 생기기 쉬워요.

`TypedDict`는 **딕셔너리의 키와 각 값의 타입을 미리 선언**하는 방법이에요. 코드 작성 시점에 IDE가 오류를 잡아줄 수 있어요.

> 🔑 **핵심 개념**: TypedDict는 런타임(실행 중)에는 일반 dict와 동일하게 동작해요. 차이는 **정적 타입 검사** 시점(코드 작성 중, IDE)에 있어요. LangGraph는 TypedDict를 State 정의의 표준으로 사용해요.

비유하자면, 일반 `dict`는 **아무 물건이나 넣을 수 있는 서랍장**이에요. TypedDict는 **칸마다 라벨이 붙은 정리함**이에요 — "양말 칸", "속옷 칸"처럼 각 칸에 뭘 넣을지 미리 정해놨어요. 실제로 넣을 때 강제하진 않지만, 라벨을 보고 실수를 줄일 수 있어요.

> 🎯 **강의 포인트**: IDE에서 직접 보여주면 효과적이에요. TypedDict로 정의한 State에서 존재하지 않는 키를 접근하면 IDE가 빨간 줄로 오류를 표시해요.

In [1]:
# ---------------------------------------------------
# TypedDict vs dict 비교
# ---------------------------------------------------
# Python 3.8+에서는 typing 모듈에서, 3.9+에서는 typing_extensions에서 가져와요
from typing import Dict, TypedDict

# ---------------------------------------------------
# 1) 일반 dict: 타입 제약 없음
# ---------------------------------------------------
sample_dict: Dict[str, str] = {
    "name": "Alice",
    "age": "30",     # 문자열로 저장해도 오류 없음
    "job": "개발자",
}

# dict는 자유롭게 타입 변경, 새 키 추가가 가능해요
sample_dict["age"] = 30           # 타입이 바뀌어도 오류 없음
sample_dict["country"] = "Korea"  # 새로운 키 추가도 가능

print("일반 dict:", sample_dict)
print("타입:", type(sample_dict))

일반 dict: {'name': 'Alice', 'age': 30, 'job': '개발자', 'country': 'Korea'}
타입: <class 'dict'>


In [2]:
# ---------------------------------------------------
# 2) TypedDict: 구조를 미리 선언
# ---------------------------------------------------
# TypedDict로 정의하면 각 키의 타입이 고정돼요
class Person(TypedDict):
    name: str    # 이름은 반드시 문자열
    age: int     # 나이는 반드시 정수
    job: str     # 직업은 반드시 문자열

# 올바른 사용
typed_person: Person = {
    "name": "Alice",
    "age": 30,
    "job": "개발자",
}

print("TypedDict:", typed_person)
print("타입:", type(typed_person))  # 런타임에는 여전히 dict예요!

# 런타임에는 일반 dict와 동일하게 값에 접근해요
print(f"이름: {typed_person['name']}, 나이: {typed_person['age']}")

TypedDict: {'name': 'Alice', 'age': 30, 'job': '개발자'}
타입: <class 'dict'>
이름: Alice, 나이: 30


In [3]:
# ---------------------------------------------------
# LangGraph에서 State 정의 패턴
# ---------------------------------------------------
# 실제 LangGraph에서는 이런 형태로 State를 정의해요
# (아직 langgraph는 import하지 않고, TypedDict만 사용해요)

class AgentState(TypedDict):
    """간단한 에이전트 상태 예시"""
    query: str        # 사용자 질문
    answer: str       # 에이전트 답변
    step: int         # 현재 실행 단계
    is_done: bool     # 완료 여부

# 초기 상태 생성
state: AgentState = {
    "query": "LangGraph란 무엇인가요?",
    "answer": "",
    "step": 0,
    "is_done": False,
}

print("초기 상태:", state)
print("쿼리:", state["query"])
print("완료 여부:", state["is_done"])

초기 상태: {'query': 'LangGraph란 무엇인가요?', 'answer': '', 'step': 0, 'is_done': False}
쿼리: LangGraph란 무엇인가요?
완료 여부: False


---

## 2. Annotated — 타입에 메타데이터 붙이기

### 왜 리듀서가 필요할까요?

LangGraph에서 여러 노드가 같은 State 필드를 수정해요. 예를 들어, 노드 A가 메시지를 추가하고, 노드 B도 메시지를 추가해요. 만약 단순 덮어쓰기라면 노드 B의 메시지만 남고 노드 A의 메시지는 사라져요! **리듀서(reducer)**는 "이 필드를 업데이트할 때 덮어쓰지 말고 이렇게 합쳐라"는 규칙을 정의해요. 그리고 이 규칙을 타입 힌트에 붙이는 도구가 바로 `Annotated`예요.

`Annotated`는 Python 3.9에서 도입된 타입 힌트 도구예요. **기존 타입에 추가 정보(메타데이터)**를 붙일 수 있게 해줘요.

기본 문법은 이렇게 생겼어요:
```python
변수명: Annotated[원래_타입, 메타데이터]
```

LangGraph에서 `Annotated`의 핵심 용도는 **리듀서(reducer) 함수를 State 필드에 연결**하는 거예요:
```python
messages: Annotated[list, add_messages]  # add_messages가 리듀서
```

> 🔑 **핵심 개념**: `Annotated`의 메타데이터는 런타임에 무시돼요. LangGraph는 State를 처리할 때 이 메타데이터를 읽어서 어떤 리듀서를 사용할지 결정해요. 즉, **LangGraph에게 "이 필드를 업데이트할 때 이 함수를 사용해"** 라고 알려주는 거예요.

> 💡 **실무 팁**: Annotated 없이 State 필드를 정의하면, 그 필드는 업데이트될 때마다 **덮어쓰기(overwrite)**가 일어나요. messages처럼 누적해야 하는 필드는 반드시 `Annotated[list, add_messages]`로 선언해야 해요.

In [4]:
# ---------------------------------------------------
# Annotated 기본 사용법
# ---------------------------------------------------
from typing import Annotated

# 타입에 설명(문서화) 메타데이터를 붙여요
name: Annotated[str, "사용자 이름 (2~50자)"] = "Alice"
age: Annotated[int, "나이 (0~150 범위)"] = 30

print(f"이름: {name}")
print(f"나이: {age}")

# Annotated의 메타데이터에 접근하는 방법
import typing
hints = typing.get_type_hints(lambda: None, include_extras=True)

# Annotated[str, "설명"]에서 메타데이터 직접 확인
age_type = Annotated[int, "나이 (0~150 범위)"]
print(f"\nAnnotated 구조: {age_type}")
print(f"원래 타입: {typing.get_args(age_type)[0]}")
print(f"메타데이터: {typing.get_args(age_type)[1]}")

이름: Alice
나이: 30

Annotated 구조: typing.Annotated[int, '나이 (0~150 범위)']
원래 타입: <class 'int'>
메타데이터: 나이 (0~150 범위)


In [2]:
# ---------------------------------------------------
# LangGraph에서 Annotated + 리듀서 패턴
# ---------------------------------------------------
# 이 패턴이 LangGraph에서 가장 중요해요!
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages  # LangGraph 리듀서 함수

class ChatState(TypedDict):
    """채팅 에이전트의 상태 정의"""
    # add_messages 리듀서: 새 메시지를 기존 리스트에 추가(누적)
    messages: Annotated[list, add_messages]
    # 일반 필드: 업데이트 시 덮어쓰기
    user_name: str
    session_id: str

# ChatState 정의 완료!
print("messages 필드 타입 힌트:", ChatState.__annotations__["messages"])
#   -> Annotated[list, add_messages]
#   -> 노드가 messages를 반환하면 add_messages가 기존 리스트에 추가해요

messages 필드 타입 힌트: typing.Annotated[list, <function _add_messages_wrapper.<locals>._add_messages at 0x113e61e80>]


In [3]:
# ---------------------------------------------------
# Annotated 없는 필드 vs. Annotated + 리듀서 필드 비교
# ---------------------------------------------------
# 두 가지 업데이트 방식을 직접 비교해볼게요

# 케이스 1: 일반 필드 - 업데이트 시 덮어쓰기
class OverwriteState(TypedDict):
    count: int  # Annotated 없음 → 덮어쓰기

# 케이스 2: Annotated + 리듀서 - 업데이트 시 누적
def append_reducer(left: list, right: list) -> list:
    """간단한 누적 리듀서 예시: 두 리스트를 합쳐요"""
    return left + right

class AccumulateState(TypedDict):
    items: Annotated[list, append_reducer]  # 리듀서로 누적

# ---------------------------------------------------
# OverwriteState로 '덮어쓰기' 동작 확인
# ---------------------------------------------------
# 리듀서가 없는 필드는 새 값이 들어오면 그대로 덮어써요
overwrite_state: OverwriteState = {"count": 1}
print("OverwriteState 초기:", overwrite_state)

# 노드가 새 값을 반환했다고 가정 (단순 대입 = 덮어쓰기)
overwrite_state["count"] = 5
print("OverwriteState 업데이트 후:", overwrite_state)
# → 이전 값 1은 사라지고 5만 남아요

# ---------------------------------------------------
# AccumulateState로 '누적' 동작 확인
# ---------------------------------------------------
# 리듀서가 붙은 필드는 새 값이 기존 값과 병합돼요
accumulate_state: AccumulateState = {"items": ["item1", "item2"]}
print("\nAccumulateState 초기:", accumulate_state)

# 노드가 새 items를 반환했다고 가정 → LangGraph가 append_reducer를 자동 호출
# (여기서는 직접 호출해서 동일한 효과를 시뮬레이션해요)
accumulate_state["items"] = append_reducer(
    accumulate_state["items"], ["item3", "item4"]
)
print("AccumulateState 업데이트 후:", accumulate_state)
# → 기존 데이터가 유지되면서 새 항목이 뒤에 누적돼요

# 한 번 더 누적해도 이전 값이 사라지지 않아요
accumulate_state["items"] = append_reducer(
    accumulate_state["items"], ["item5"]
)
print("AccumulateState 한 번 더:", accumulate_state)

OverwriteState 초기: {'count': 1}
OverwriteState 업데이트 후: {'count': 5}

AccumulateState 초기: {'items': ['item1', 'item2']}
AccumulateState 업데이트 후: {'items': ['item1', 'item2', 'item3', 'item4']}
AccumulateState 한 번 더: {'items': ['item1', 'item2', 'item3', 'item4', 'item5']}


---

## 3. Pydantic BaseModel — 데이터 유효성 검사

Pydantic은 Python 데이터 유효성 검사 라이브러리예요. LangGraph에서는 노드의 입력/출력 데이터나 도구(tool)의 파라미터를 정의할 때 많이 사용해요.

TypedDict가 **구조를 정의**한다면, Pydantic BaseModel은 **값의 제약 조건까지 검사**해요.

| 기능 | TypedDict | Pydantic BaseModel |
|------|-----------|--------------------|
| 키 타입 선언 | O | O |
| 값 제약 조건 (min, max 등) | X | O |
| 런타임 유효성 검사 | X | O |
| 자동 타입 변환 | X | O |
| LangGraph State 정의 | 주로 사용 | 가능 |

> 🎯 **강의 포인트**: LangGraph에서 도구(tool)를 정의할 때 Pydantic을 자주 사용해요. `@tool` 데코레이터로 만든 함수의 파라미터가 Pydantic 스키마로 변환돼서 LLM에게 전달돼요.

> ⚠️ **자주 하는 실수**: Pydantic V1의 `min_items`, `max_items` 같은 validator는 V2에서 변경됐어요. V2에서는 `Field(min_length=..., max_length=...)` 형태를 사용해요.

In [7]:
# ---------------------------------------------------
# Pydantic BaseModel 기본 사용법 (V2 문법)
# ---------------------------------------------------
from pydantic import BaseModel, Field, ValidationError

class Employee(BaseModel):
    """직원 정보 모델"""
    id: int = Field(description="직원 고유 ID")
    name: str = Field(min_length=2, max_length=50, description="직원 이름 (2~50자)")
    age: int = Field(gt=18, lt=65, description="나이 (19~64세)")
    salary: float = Field(gt=0, description="급여 (양수)")
    department: str = Field(default="일반", description="부서명")

# 올바른 데이터로 생성
emp = Employee(
    id=1,
    name="김철수",
    age=30,
    salary=5000000.0,
    department="개발팀"
)

print("직원 정보:", emp)
print("이름:", emp.name)
print("부서:", emp.department)

직원 정보: id=1 name='김철수' age=30 salary=5000000.0 department='개발팀'
이름: 김철수
부서: 개발팀


In [8]:
# ---------------------------------------------------
# 유효성 검사 실패 시 에러 처리
# ---------------------------------------------------
# 잘못된 데이터를 넣으면 ValidationError가 발생해요
try:
    invalid_emp = Employee(
        id=2,
        name="A",      # 너무 짧음 (min_length=2 위반)
        age=17,        # 너무 어림 (gt=18 위반)
        salary=-100,   # 음수 (gt=0 위반)
    )
except ValidationError as e:
    # 유효성 검사 실패!
    print("에러 수:", len(e.errors()))
    for error in e.errors():
        print(f"  - 필드: {error['loc']}, 오류: {error['msg']}")

에러 수: 3
  - 필드: ('name',), 오류: String should have at least 2 characters
  - 필드: ('age',), 오류: Input should be greater than 18
  - 필드: ('salary',), 오류: Input should be greater than 0


In [9]:
# ---------------------------------------------------
# Pydantic + Annotated 조합 (V2 스타일)
# ---------------------------------------------------
# Pydantic V2에서는 Annotated와 Field를 조합하는 방식도 있어요
from typing import Annotated
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    """LangGraph 도구에서 사용하는 검색 쿼리 스키마 예시"""
    query: Annotated[str, Field(min_length=1, description="검색 쿼리 문자열")]
    max_results: Annotated[int, Field(gt=0, le=10, default=5, description="최대 결과 수 (1~10)")]
    language: Annotated[str, Field(default="ko", description="검색 언어")]

# 기본값이 있는 필드는 생략 가능해요
search = SearchQuery(query="LangGraph 튜토리얼")
print("검색 쿼리:", search)
print("JSON 형태:", search.model_dump())
# (max_results, language는 기본값이 적용됐어요)

검색 쿼리: query='LangGraph 튜토리얼' max_results=5 language='ko'
JSON 형태: {'query': 'LangGraph 튜토리얼', 'max_results': 5, 'language': 'ko'}


---

## 4. add_messages 리듀서

`add_messages`는 LangGraph에서 제공하는 핵심 리듀서 함수예요. 채팅 애플리케이션에서 대화 기록을 관리할 때 사용해요.

### 동작 규칙

| 상황 | 동작 |
|------|------|
| 새 메시지 ID가 없거나 기존과 다름 | 기존 리스트에 **추가** |
| 새 메시지의 ID가 기존 메시지와 같음 | 기존 메시지를 **교체** |

이 규칙 덕분에 에이전트가 이전 메시지를 수정해야 할 때도 유연하게 처리할 수 있어요.

> 🔑 **핵심 개념**: `add_messages(left, right)`는 두 개의 메시지 리스트를 받아요. `left`는 현재 State의 messages, `right`는 노드가 반환한 새 messages예요. LangGraph가 자동으로 이 함수를 호출해서 상태를 업데이트해요.

비유하자면, `add_messages`는 **대화 녹음기**와 비슷해요. 새 발언이 들어오면 기존 녹음 뒤에 이어 붙이고(추가), 이미 녹음된 발언을 수정하고 싶으면 같은 ID의 발언을 새 버전으로 교체해요.

> ⚠️ **자주 하는 실수**: `add_messages`를 사용할 때 메시지에 `id`를 지정하지 않으면 항상 추가만 돼요. ID 기반 교체가 필요한 경우(예: 임시 응답을 최종 응답으로 바꿀 때) 반드시 동일한 `id`를 지정해야 해요.

> 💡 **실무 팁**: 대화 기록이 너무 길어지면 컨텍스트 창(context window) 한계에 걸릴 수 있어요. Part 4에서 배울 `DeleteMessages`나 요약(Summarization) 기법으로 메시지 수를 관리해요.

In [10]:
# ---------------------------------------------------
# add_messages 기본 동작: 메시지 추가
# ---------------------------------------------------
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages

# 기존 메시지 리스트 (현재 State의 messages)
existing_messages = [
    HumanMessage(content="안녕하세요!", id="msg-1"),
]

# 새로 추가할 메시지 (노드가 반환한 messages)
new_messages = [
    AIMessage(content="안녕하세요! 무엇을 도와드릴까요?", id="msg-2"),
]

# add_messages 호출: 두 리스트를 병합해요
result = add_messages(existing_messages, new_messages)

# 병합 결과:
for msg in result:
    print(f"  [{msg.type}] id={msg.id}: {msg.content}")

  [human] id=msg-1: 안녕하세요!
  [ai] id=msg-2: 안녕하세요! 무엇을 도와드릴까요?


In [11]:
# ---------------------------------------------------
# add_messages 교체 동작: 같은 ID면 덮어써요
# ---------------------------------------------------
# 동일한 ID를 가진 메시지가 오면 교체(replace)가 일어나요

# 기존 메시지
messages_before = [
    HumanMessage(content="날씨 알려줘", id="user-1"),
    AIMessage(content="잠시만요...", id="ai-1"),  # 임시 응답
]

# 같은 ID 'ai-1'로 업데이트된 메시지
updated_message = [
    AIMessage(content="오늘 서울 날씨는 맑고 23도예요!", id="ai-1"),  # 교체!
]

result = add_messages(messages_before, updated_message)

# 교체 결과 (리스트 길이 확인):
print(f"  이전: {len(messages_before)}개 → 이후: {len(result)}개")
# 업데이트된 메시지:
for msg in result:
    print(f"  [{msg.type}] id={msg.id}: {msg.content}")

  이전: 2개 → 이후: 2개
  [human] id=user-1: 날씨 알려줘
  [ai] id=ai-1: 오늘 서울 날씨는 맑고 23도예요!


In [12]:
# ---------------------------------------------------
# add_messages를 실제 State에 적용한 예시
# ---------------------------------------------------
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain.messages import HumanMessage, AIMessage

# LangGraph State 정의 (이 패턴을 Part 2부터 계속 사용해요)
class ConversationState(TypedDict):
    """대화 상태 - LangGraph의 가장 기본적인 State 패턴"""
    messages: Annotated[list, add_messages]  # 누적되는 메시지 리스트

# 초기 상태
state: ConversationState = {"messages": []}
print("초기 상태:", state)

# 시뮬레이션: 사용자 메시지 추가
# (실제 LangGraph에서는 노드가 {"messages": [...]}를 반환하면 자동으로 add_messages가 호출돼요)
turn1 = [HumanMessage(content="LangGraph를 어떻게 시작하나요?", id="h1")]
state["messages"] = add_messages(state["messages"], turn1)

turn2 = [AIMessage(content="langgraph 패키지를 설치하고 StateGraph를 정의하면 돼요!", id="a1")]
state["messages"] = add_messages(state["messages"], turn2)

print(f"\n대화 기록 ({len(state['messages'])}개 메시지):")
for msg in state["messages"]:
    print(f"  [{msg.type:5s}] {msg.content}")

초기 상태: {'messages': []}

대화 기록 (2개 메시지):
  [human] LangGraph를 어떻게 시작하나요?
  [ai   ] langgraph 패키지를 설치하고 StateGraph를 정의하면 돼요!


---

## 5. 종합 실습 — 모든 개념을 하나로

지금까지 배운 TypedDict, Annotated, Pydantic, add_messages를 모두 조합해서 실제 LangGraph State처럼 작성해볼게요.

> 🎯 **강의 포인트**: 아래 코드는 Part 2에서 실제로 만들 챗봇의 State 정의와 거의 동일해요. 이 패턴을 완전히 이해하면 이후 모든 노트북이 쉬워져요.

In [13]:
# ---------------------------------------------------
# 종합: TypedDict + Annotated + add_messages 완전 패턴
# ---------------------------------------------------
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages

# 1. Pydantic으로 도구 입력 스키마 정의
class SearchInput(BaseModel):
    """웹 검색 도구의 입력 스키마"""
    query: str = Field(min_length=1, description="검색어")
    num_results: int = Field(default=3, gt=0, le=5, description="결과 수")

# 2. TypedDict + Annotated로 State 정의
class AgentState(TypedDict):
    """에이전트 상태 - 완전한 LangGraph 패턴"""
    messages: Annotated[list, add_messages]  # 리듀서로 메시지 누적
    user_name: str                           # 덮어쓰기 (일반 필드)
    iteration: int                           # 반복 횟수 추적

# 3. 시뮬레이션 실행
state: AgentState = {
    "messages": [SystemMessage(content="당신은 도움이 되는 AI 어시스턴트예요.", id="sys-1")],
    "user_name": "학생",
    "iteration": 0,
}

# 대화 진행
state["messages"] = add_messages(
    state["messages"],
    [HumanMessage(content="TypedDict와 Pydantic의 차이가 뭔가요?", id="h-1")]
)
state["iteration"] += 1

state["messages"] = add_messages(
    state["messages"],
    [AIMessage(content="TypedDict는 구조 정의, Pydantic은 유효성 검사까지 해요!", id="a-1")]
)
state["iteration"] += 1

# ==================================================
print(f"사용자: {state['user_name']}")
print(f"반복 횟수: {state['iteration']}")
print(f"메시지 수: {len(state['messages'])}")
# ==================================================
for msg in state["messages"]:
    label = {"human": "사용자", "ai": "AI", "system": "시스템"}.get(msg.type, msg.type)
    print(f"[{label}] {msg.content}")

사용자: 학생
반복 횟수: 2
메시지 수: 3
[시스템] 당신은 도움이 되는 AI 어시스턴트예요.
[사용자] TypedDict와 Pydantic의 차이가 뭔가요?
[AI] TypedDict는 구조 정의, Pydantic은 유효성 검사까지 해요!


In [14]:
# ============================================================
# 구현 예시: 나만의 AgentState 만들어보기
# ============================================================
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain.messages import HumanMessage, AIMessage


def todo_reducer(left: list[str], right: list[str]) -> list[str]:
    """할 일 목록 리듀서: 순서를 유지하면서 중복 없이 합쳐요."""
    merged: list[str] = []
    for item in [*left, *right]:
        if item not in merged:
            merged.append(item)
    return merged


class TodoState(TypedDict):
    messages: Annotated[list, add_messages]
    todos: Annotated[list[str], todo_reducer]
    current_task: str


# 리듀서 동작 테스트
state: TodoState = {
    "messages": [HumanMessage(content="LangGraph 실습을 시작해요")],
    "todos": [],
    "current_task": "상태 설계",
}

state["todos"] = todo_reducer(state["todos"], ["LangGraph 공부", "실습 완료"])
state["todos"] = todo_reducer(state["todos"], ["실습 완료", "복습하기"])
state["messages"] = add_messages(
    state["messages"],
    [AIMessage(content="중복 없이 TODO 목록을 누적했어요")],
)

print("할 일 목록:", state["todos"])
print("현재 작업:", state["current_task"])
print("메시지 수:", len(state["messages"]))


할 일 목록: ['LangGraph 공부', '실습 완료', '복습하기']
현재 작업: 상태 설계
메시지 수: 2


---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **TypedDict**: 딕셔너리의 키와 타입을 미리 선언해요. 런타임에는 일반 dict와 동일하지만, IDE와 타입 검사기가 오류를 미리 잡아줘요. LangGraph State 정의의 표준이에요.

- **Annotated**: 타입 힌트에 메타데이터를 붙이는 Python 도구예요. LangGraph에서는 `Annotated[list, add_messages]` 형태로 리듀서 함수를 State 필드에 연결할 때 사용해요.

- **Pydantic BaseModel**: 런타임 유효성 검사를 제공하는 라이브러리예요. `Field`로 최솟값, 최댓값, 설명 등을 지정할 수 있어요. V2에서는 `min_length`, `max_length`, `gt`, `lt` 파라미터를 사용해요.

- **add_messages**: LangGraph 내장 리듀서 함수예요. `(left, right)` 두 메시지 리스트를 받아서 병합해요. 같은 ID의 메시지가 있으면 교체, 없으면 추가해요. 대화 기록 누적에 필수예요.

## 다음 노트북 예고

다음 `02-Models.ipynb`에서는 **LangGraph에서 LLM 모델을 초기화하고 사용하는 방법**을 배워요. `init_chat_model`로 OpenAI, Anthropic, Google 등 다양한 모델을 통일된 인터페이스로 다루고, invoke/stream/batch 호출 패턴을 실습해요.